# Moon AI — Moon Chat v0 Student Pilot Training Notebook
**Chief ML Systems Architect:** Prem Gaikwad (India 🇮🇳)  
**Ownership Policy:** `TEACHER_STUDENT_RANDOM_INIT` (Policy B)  
**Target Model:** `moon-chat-pilot-25m` (25,698,816 parameters, 8 layers, 8 heads, 512 embd)  
**Hardware Target:** Google Colab NVIDIA Tesla T4 (15.3 GB VRAM)  

--- 
### Training Mission
Execute the first scratch-initialized student pilot training run on the expanded **572-record / 176,738-token corpus**.

Key Invariants:
1. **Policy B Invariant:** Student model initializes strictly from Gaussian random scratch ($N(0, 0.02^2)$). Zero pretrained checkpoint weights transferred.
2. **Zero External APIs:** 100% offline dataset and sovereign training execution.
3. **Assistant-Only Masking:** Loss is computed strictly on assistant response tokens (prompts masked with -1).
4. **Verified Manifests:** Uses `expanded_chat_train.jsonl` (468 records), `expanded_chat_val.jsonl` (52 records), `expanded_chat_test.jsonl` (52 records).

## 1. Verify GPU Hardware

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:    {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"Total VRAM:     {props.total_memory / (1024**3):.2f} GB ({props.total_memory / (1024**2):.1f} MB)")

## 2. Check Readiness Gate

In [ ]:
import sys
sys.path.insert(0, '.')
from training.chat.readiness_gate import ChatReadinessGate

gate = ChatReadinessGate(
    train_manifest="datasets/manifests/expanded_chat_train.jsonl",
    val_manifest="datasets/manifests/expanded_chat_val.jsonl",
    test_manifest="datasets/manifests/expanded_chat_test.jsonl"
)
res = gate.run_all_checks()
print(f"Gate Status:    {res['status']}")
print(f"Passed:         {res['passed']}")
print(f"Blockers:       {len(res['blockers'])}")
assert res['passed'], f"Readiness gate blocked: {res['blockers']}"

## 3. Launch Student Pilot Training Run (500 Iterations)

In [ ]:
from training.chat.train import train_moon_chat

results = train_moon_chat(
    config_path="training/chat/pilot_config.yaml",
    max_iters_override=500
)
print("Training Complete!")
print(f"Status:           {results['status']}")
print(f"Final Val Loss:   {results['final_val_loss']:.4f}")
print(f"Perplexity:       {results['perplexity']:.2f}")
print(f"Checkpoint Path:  {results['checkpoint_path']}")
print(f"Checkpoint SHA:   {results['checkpoint_sha256']}")

## 4. Test Text Generation from Trained Checkpoint

In [ ]:
from training.chat.sample import load_chat_model, generate_chat_response
from datasets.chat.tokenizer import SovereignTokenizer

tokenizer = SovereignTokenizer.load("models/chat/tokenizer.json")
model, ckpt = load_chat_model("models/checkpoints/chat/moon_chat_pilot_v0.pt")

for prompt in ["Hi", "What is a function in programming?", "Explain photosynthesis."]:
    response = generate_chat_response(model, tokenizer, prompt, max_new_tokens=40)
    print(f"=== Prompt: {prompt} ===")
    print(response)
    print()
